## Завдання 1. Setup і завантаження

In [1]:
!pip install -q pgserver==0.1.4 \
    --python-version 3.12 \
    --only-binary=:all: \
    --target=/usr/local/lib/python3.13/dist-packages

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 5.0 MB/s eta 0:00:00


In [2]:
!pip install -q pgserver psycopg2-binary sqlalchemy pandas pyarrow ucimlrepo

import pandas as pd
import pgserver
from sqlalchemy import create_engine, text

pg = pgserver.get_server('/tmp/hw3_pg', cleanup_mode='stop')
engine = create_engine(pg.get_uri(), future=True)

with engine.connect() as conn:
    version = conn.execute(text('SELECT version()')).scalar_one()
    print(version)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 35.2 MB/s eta 0:00:00
PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


In [3]:
from ucimlrepo import fetch_ucirepo

# UCI Seoul Bike Sharing Demand
dataset = fetch_ucirepo(id=560)

df = pd.concat(
    [dataset.data.features, dataset.data.targets],
    axis=1
)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

df.head()

Shape: (8760, 14)

Columns:
['Date', 'Rented Bike Count', 'Hour', 'Temperature', 'Humidity', 'Wind speed', 'Visibility', 'Dew point temperature', 'Solar Radiation', 'Rainfall', 'Snowfall', 'Seasons', 'Holiday', 'Functioning Day']


,Date,Rented Bike Count,Hour,Temperature,Humidity,Wind speed,Visibility,Dew point temperature,Solar Radiation,Rainfall,Snowfall,Seasons,Holiday,Functioning Day
0,1/12/2017,254,0,-5.2,37,2.2,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
1,1/12/2017,204,1,-5.5,38,0.8,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
2,1/12/2017,173,2,-6.0,39,1.0,2000,-17.7,0.0,0.0,0.0,Winter,No Holiday,Yes
3,1/12/2017,107,3,-6.2,40,0.9,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
4,1/12/2017,78,4,-6.0,36,2.3,2000,-18.6,0.0,0.0,0.0,Winter,No Holiday,Yes


In [4]:
csv_path = "/content/hw3_seoul_bike.csv"

df.to_csv(csv_path, index=False)

print("CSV saved:", csv_path)
print("Rows:", len(df))

CSV saved: /content/hw3_seoul_bike.csv
Rows: 8760


In [5]:
with engine.begin() as conn:
    conn.execute(text("""
        DROP TABLE IF EXISTS hw3_bike_clean CASCADE;
        DROP TABLE IF EXISTS hw3_bike_staging CASCADE;

        CREATE TABLE hw3_bike_staging (
            date_raw TEXT,
            rented_bike_count_raw TEXT,
            hour_raw TEXT,
            temperature_raw TEXT,
            humidity_raw TEXT,
            wind_speed_raw TEXT,
            visibility_raw TEXT,
            dew_point_temperature_raw TEXT,
            solar_radiation_raw TEXT,
            rainfall_raw TEXT,
            snowfall_raw TEXT,
            seasons_raw TEXT,
            holiday_raw TEXT,
            functioning_day_raw TEXT
        );
    """))

print("Staging table created successfully.")

Staging table created successfully.


In [6]:
raw_conn = engine.raw_connection()

try:
    with raw_conn.cursor() as cur:
        with open(csv_path, "r", encoding="utf-8", newline="") as file:
            cur.copy_expert(
                """
                COPY hw3_bike_staging (
                    date_raw,
                    rented_bike_count_raw,
                    hour_raw,
                    temperature_raw,
                    humidity_raw,
                    wind_speed_raw,
                    visibility_raw,
                    dew_point_temperature_raw,
                    solar_radiation_raw,
                    rainfall_raw,
                    snowfall_raw,
                    seasons_raw,
                    holiday_raw,
                    functioning_day_raw
                )
                FROM STDIN
                WITH (
                    FORMAT csv,
                    HEADER true,
                    DELIMITER ','
                );
                """,
                file
            )

    raw_conn.commit()

finally:
    raw_conn.close()

print("CSV loaded into staging successfully.")

CSV loaded into staging successfully.


In [7]:
pd.read_sql("""
    SELECT COUNT(*) AS loaded_rows
    FROM hw3_bike_staging;
""", engine)

,loaded_rows
0,8760


In [8]:
pd.read_sql("""
    SELECT
        date_raw,
        rented_bike_count_raw,
        hour_raw,
        temperature_raw,
        seasons_raw,
        holiday_raw,
        functioning_day_raw
    FROM hw3_bike_staging
    LIMIT 5;
""", engine)

,date_raw,rented_bike_count_raw,hour_raw,temperature_raw,seasons_raw,holiday_raw,functioning_day_raw
0,1/12/2017,254,0,-5.2,Winter,No Holiday,Yes
1,1/12/2017,204,1,-5.5,Winter,No Holiday,Yes
2,1/12/2017,173,2,-6.0,Winter,No Holiday,Yes
3,1/12/2017,107,3,-6.2,Winter,No Holiday,Yes
4,1/12/2017,78,4,-6.0,Winter,No Holiday,Yes


## Завдання 2. Staging і DDL

In [9]:
with engine.begin() as conn:
    conn.execute(text("""
        DROP TABLE IF EXISTS hw3_bike_clean CASCADE;

        CREATE TABLE hw3_bike_clean (
            record_id BIGINT
                GENERATED BY DEFAULT AS IDENTITY
                PRIMARY KEY,

            date DATE NOT NULL,

            rented_bike_count INTEGER NOT NULL
                CHECK (rented_bike_count >= 0),

            hour SMALLINT NOT NULL
                CHECK (hour BETWEEN 0 AND 23),

            temperature NUMERIC(3,1),

            humidity SMALLINT
                CHECK (humidity BETWEEN 0 AND 100),

            wind_speed NUMERIC(4,1)
                CHECK (wind_speed >= 0),

            visibility SMALLINT
                CHECK (visibility >= 0),

            dew_point_temperature NUMERIC(4,1),

            solar_radiation NUMERIC(5,2)
                CHECK (solar_radiation >= 0),

            rainfall NUMERIC(5,1)
                CHECK (rainfall >= 0),

            snowfall NUMERIC(4,1)
                CHECK (snowfall >= 0),

            seasons TEXT,

            holiday BOOLEAN,

            functioning_day BOOLEAN NOT NULL
        );
    """))

print("Clean table structure created successfully.")

Clean table structure created successfully.


In [10]:
with engine.begin() as conn:
    conn.execute(text("""
        WITH parsed AS (
            SELECT
                TO_DATE(
                    NULLIF(TRIM(date_raw), ''),
                    'DD/MM/YYYY'
                ) AS date,

                NULLIF(TRIM(rented_bike_count_raw), '')::INTEGER
                    AS rented_bike_count,

                NULLIF(TRIM(hour_raw), '')::SMALLINT
                    AS hour,

                NULLIF(TRIM(temperature_raw), '')::NUMERIC
                    AS temperature,

                NULLIF(TRIM(humidity_raw), '')::SMALLINT
                    AS humidity,

                NULLIF(TRIM(wind_speed_raw), '')::NUMERIC
                    AS wind_speed,

                NULLIF(TRIM(visibility_raw), '')::SMALLINT
                    AS visibility,

                NULLIF(TRIM(dew_point_temperature_raw), '')::NUMERIC
                    AS dew_point_temperature,

                NULLIF(TRIM(solar_radiation_raw), '')::NUMERIC
                    AS solar_radiation,

                NULLIF(TRIM(rainfall_raw), '')::NUMERIC
                    AS rainfall,

                NULLIF(TRIM(snowfall_raw), '')::NUMERIC
                    AS snowfall,

                NULLIF(TRIM(seasons_raw), '')
                    AS seasons,

                CASE TRIM(holiday_raw)
                    WHEN 'Holiday' THEN TRUE
                    WHEN 'No Holiday' THEN FALSE
                    ELSE NULL
                END AS holiday,

                CASE TRIM(functioning_day_raw)
                    WHEN 'Yes' THEN TRUE
                    WHEN 'No' THEN FALSE
                    ELSE NULL
                END AS functioning_day

            FROM hw3_bike_staging
        )

        INSERT INTO hw3_bike_clean (
            date,
            rented_bike_count,
            hour,
            temperature,
            humidity,
            wind_speed,
            visibility,
            dew_point_temperature,
            solar_radiation,
            rainfall,
            snowfall,
            seasons,
            holiday,
            functioning_day
        )

        SELECT
            date,
            rented_bike_count,
            hour,
            temperature,
            humidity,
            wind_speed,
            visibility,
            dew_point_temperature,
            solar_radiation,
            rainfall,
            snowfall,
            seasons,
            holiday,
            functioning_day
        FROM parsed;
    """))

print("Clean data inserted successfully.")

Clean data inserted successfully.


## Завдання 3. Data-quality audit

In [11]:
pd.read_sql("""

    SELECT 'staging' AS layer, COUNT(*) AS n_rows
    FROM hw3_bike_staging

    UNION ALL

    SELECT 'clean' AS layer, COUNT(*) AS n_rows
    FROM hw3_bike_clean

""", engine)

,layer,n_rows
0,staging,8760
1,clean,8760


In [12]:
pd.read_sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT (hour, date)) AS distinct_business_keys,
        COUNT(*) - COUNT(DISTINCT (hour, date)) AS duplicate_rows
    FROM hw3_bike_clean;

""", engine)

,total_rows,distinct_business_keys,duplicate_rows
0,8760,8760,0


In [13]:
pd.read_sql("""
    SELECT
        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(date_raw), '?') IS NULL
        ) AS missing_date,

        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(hour_raw), '?') IS NULL
        ) AS missing_hour,


        COUNT(*) FILTER (
              WHERE NULLIF (TRIM(holiday_raw), '?') IS NULL
        ) AS missing_holiday,


        COUNT(*) FILTER (
              WHERE NULLIF(TRIM(functioning_day_raw), '?') IS NULL
        ) AS missing_functioning_day,


        COUNT(*) FILTER (
              WHERE NULLIF(TRIM(seasons_raw), '?') IS NULL
        ) AS missing_seasons

        FROM hw3_bike_staging;

""", engine)

,missing_date,missing_hour,missing_holiday,missing_functioning_day,missing_seasons
0,0,0,0,0,0


In [14]:
pd.read_sql("""
      SELECT
          ROUND(COUNT(*) FILTER (
              WHERE temperature IS NULL)* 100.0 / NULLIF(COUNT(*), 0),2
          ) as pct_null_temperature,

          ROUND(COUNT(*) FILTER (
              WHERE humidity IS NULL)* 100.0 / NULLIF(COUNT(*), 0),2
          ) as pct_null_humidity,

          ROUND(COUNT(*) FILTER (
              WHERE wind_speed IS NULL)* 100.0 / NULLIF(COUNT(*), 0),2
          ) as pct_null_wind_speed,

          ROUND(COUNT(*) FILTER (
              WHERE rainfall IS NULL)* 100.0 / NULLIF(COUNT(*), 0),2
          ) as pct_null_rainfall,

          ROUND(COUNT(*) FILTER (
              WHERE holiday IS NULL)* 100.0 / NULLIF(COUNT(*), 0),2
          ) as pct_null_holiday,

          ROUND(COUNT(*) FILTER (
              WHERE functioning_day IS NULL)* 100.0 / NULLIF(COUNT(*), 0),2
          ) as pct_null_functioning_day,

          COUNT(*) AS total_rows

      FROM hw3_bike_clean;

""", engine)

,pct_null_temperature,pct_null_humidity,pct_null_wind_speed,pct_null_rainfall,pct_null_holiday,pct_null_functioning_day,total_rows
0,0.0,0.0,0.0,0.0,0.0,0.0,8760


In [15]:
pd.read_sql("""
    SELECT distinct seasons_raw
    FROM hw3_bike_staging

""", engine)

,seasons_raw
0,Spring
1,Winter
2,Summer
3,Autumn


In [16]:
pd.read_sql("""
    SELECT distinct holiday_raw
    FROM hw3_bike_staging

""", engine)

,holiday_raw
0,Holiday
1,No Holiday


In [17]:
pd.read_sql("""
    SELECT distinct functioning_day_raw
    FROM hw3_bike_staging

""", engine)

,functioning_day_raw
0,No
1,Yes


In [18]:
pd.read_sql("""
    SELECT
        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(date_raw), '') IS NULL
        ) AS missing_date,

        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(hour_raw), '') IS NULL
        ) AS missing_hour,


        COUNT(*) FILTER (
              WHERE NULLIF (TRIM(holiday_raw), '') IS NULL
        ) AS missing_holiday,


        COUNT(*) FILTER (
              WHERE NULLIF(TRIM(functioning_day_raw), '') IS NULL
        ) AS missing_functioning_day,


        COUNT(*) FILTER (
              WHERE NULLIF(TRIM(seasons_raw), '') IS NULL
        ) AS missing_seasons

        FROM hw3_bike_staging;

""", engine)

,missing_date,missing_hour,missing_holiday,missing_functioning_day,missing_seasons
0,0,0,0,0,0


## Завдання 4. Розподіли та потенційні аномалії

In [19]:
pd.read_sql("""
    SELECT
        seasons,
        COUNT(*) AS n_rows,
        ROUND(
            COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (),
            2
        ) AS pct_rows
    FROM hw3_bike_clean
    GROUP BY seasons
    ORDER BY n_rows DESC, seasons
    LIMIT 10;
""", engine)

,seasons,n_rows,pct_rows
0,Spring,2208,25.21
1,Summer,2208,25.21
2,Autumn,2184,24.93
3,Winter,2160,24.66


In [20]:
pd.read_sql("""
    SELECT
        COUNT(rented_bike_count) AS n_non_null,
        MIN(rented_bike_count) AS min_value,
        MAX(rented_bike_count) AS max_value,
        ROUND(AVG(rented_bike_count)::NUMERIC, 2) AS avg_value,
        ROUND(STDDEV_SAMP(rented_bike_count)::NUMERIC, 2) AS sd_value
    FROM hw3_bike_clean;
""", engine)

,n_non_null,min_value,max_value,avg_value,sd_value
0,8760,0,3556,704.6,645.0


In [21]:
pd.read_sql("""

WITH stats AS (
    SELECT
        AVG(rented_bike_count) AS mean_value,
        STDDEV_SAMP(rented_bike_count) AS sd_value
    FROM hw3_bike_clean
    WHERE rented_bike_count IS NOT NULL
)
SELECT
    t.*,
    ROUND(
        ((t.rented_bike_count - s.mean_value) / NULLIF(s.sd_value, 0))::NUMERIC,
        2
    ) AS z_score
FROM hw3_bike_clean AS t
CROSS JOIN stats AS s
WHERE t.rented_bike_count IS NOT NULL
  AND s.sd_value > 0
  AND ABS(t.rented_bike_count - s.mean_value) > 3 * s.sd_value
ORDER BY ABS(t.rented_bike_count - s.mean_value) DESC
LIMIT 20;
""", engine)

,record_id,date,rented_bike_count,hour,temperature,humidity,wind_speed,visibility,dew_point_temperature,solar_radiation,rainfall,snowfall,seasons,holiday,functioning_day,z_score
0,4819,2018-06-19,3556,18,24.1,57,2.9,1301,15.0,0.56,0.0,0.0,Summer,False,True,4.42
1,4867,2018-06-21,3418,18,27.8,43,3.0,1933,14.0,1.35,0.0,0.0,Summer,False,True,4.21
2,4651,2018-06-12,3404,18,24.9,53,3.6,2000,14.6,1.28,0.0,0.0,Summer,False,True,4.19
3,4843,2018-06-20,3384,18,27.0,55,3.1,1246,17.1,1.26,0.0,0.0,Summer,False,True,4.15
4,4459,2018-06-04,3380,18,24.4,48,1.9,1998,12.6,0.56,0.0,0.0,Summer,False,True,4.15
5,4891,2018-06-22,3365,18,29.3,27,3.4,1977,8.3,1.24,0.0,0.0,Summer,False,True,4.12
6,4555,2018-06-08,3309,18,26.2,54,2.2,1183,16.1,0.88,0.0,0.0,Summer,False,True,4.04
7,6812,2018-09-10,3298,18,25.9,42,1.1,2000,11.9,0.48,0.0,0.0,Autumn,False,True,4.02
8,6980,2018-09-17,3277,18,25.3,56,2.8,1992,15.8,0.54,0.0,0.0,Autumn,False,True,3.99
9,6860,2018-09-12,3256,18,27.0,44,1.4,2000,13.6,0.62,0.0,0.0,Autumn,False,True,3.96


## Завдання 5. Десять DQL/EDA-запитів

**Business-question #1:** У які години найбільше орендують велосипеди?

In [22]:
pd.read_sql("""
    SELECT
        hour,
        SUM(rented_bike_count) AS total_rentals
    FROM hw3_bike_clean
    GROUP BY hour
    ORDER BY total_rentals DESC
    LIMIT 10

""", engine)

,hour,total_rentals
0,18,548568
1,19,436229
2,17,415556
3,20,390172
4,21,376479
5,8,370731
6,16,339677
7,22,336821
8,15,302653
9,14,276971


Найвищий попит на оренду велосипедів спостерігається о 18:00 — 548 568 оренд. Далі йдуть 19:00 та 17:00. Ще один помітний пік припадає на 08:00. Така закономірність показує, що використання велосипедів значною мірою пов’язане з ранковими та вечірніми годинами поїздок на роботу та з роботи.

**Business-question #2:** Як змінюється попит на велосипеди залежно від пори року?

In [23]:
pd.read_sql("""
    SELECT
        seasons,
        SUM(rented_bike_count) as sum_rent,
        ROUND(AVG(rented_bike_count),2) as avg_rent
    FROM hw3_bike_clean
        GROUP BY seasons
        ORDER BY sum_rent DESC

""", engine)

,seasons,sum_rent,avg_rent
0,Summer,2283234,1034.07
1,Autumn,1790002,819.60
2,Spring,1611909,730.03
3,Winter,487169,225.54


Попит на велосипеди має чітко виражену сезонність. Влітку спостерігається найбільша загальна та середня кількість оренд — у середньому 1 034,07 оренди на годину. Найнижчий середній показник припадає на зиму — 225,54 оренди на годину. Це показує, що в теплі пори року велосипеди використовуються значно активніше.

**Business-question #3:** Як відрізняється попит у святкові та звичайні дні?

In [24]:
pd.read_sql("""
    SELECT
      holiday,
      COUNT(*),
      SUM(rented_bike_count) as sum_rent,
      ROUND(AVG(rented_bike_count),2) as avg_rent
    FROM hw3_bike_clean
    GROUP BY holiday

""", engine)

,holiday,count,sum_rent,avg_rent
0,False,8328,5956419,715.23
1,True,432,215895,499.76


У звичайні дні попит на велосипеди вищий, ніж у святкові. Середня кількість оренд за годину становить 715,23 у звичайні дні порівняно з 499,76 у святкові. Загальна кількість оренд також значно вища у звичайні дні, хоча частково це пояснюється тим, що таких спостережень у наборі даних набагато більше.

**Business-question #4:** Чи є записи з нульовою кількістю оренд у робочі дні та в які години вони виникають?

In [25]:
pd.read_sql("""
  SELECT
    date,
    hour,
    rented_bike_count,
    functioning_day
  FROM
    hw3_bike_clean
  WHERE
      functioning_day = True AND rented_bike_count = 0

""", engine)

,date,hour,rented_bike_count,functioning_day


**Business-question #5:** У робочі дні не було знайдено записів із нульовою кількістю оренд. Тому для цієї умови немає годин, які потрібно вказувати.

In [26]:
pd.read_sql("""
    SELECT
        CASE
            WHEN hour BETWEEN 7 AND 9 THEN 'Morning peak'
            WHEN hour BETWEEN 17 AND 19 THEN 'Evening peak'
        END AS peak_period,
        COUNT(*) AS n_rows,
        SUM(rented_bike_count) AS total_rentals,
        ROUND(AVG(rented_bike_count), 2) AS avg_rentals
    FROM hw3_bike_clean
    WHERE hour BETWEEN 7 AND 9
       OR hour BETWEEN 17 AND 19
    GROUP BY peak_period
    ORDER BY avg_rentals DESC;
""", engine)

,peak_period,n_rows,total_rentals,avg_rentals
0,Evening peak,1095,1400353,1278.86
1,Morning peak,1095,827707,755.90


У вечірні години пік попит на велосипеди значно вищий, ніж у ранкові години пік. Середня кількість оренд становить 1 278,86 у період 17:00–19:00 порівняно з 755,90 у період 07:00–09:00. Це підтверджує, що вечірній період поїздок має вищий попит.

**Business-question #6:** Які записи з високим попитом припадають на звичайні дні за екстремальних температур?

In [27]:
pd.read_sql("""
    SELECT
        date,
        hour,
        temperature,
        rented_bike_count
    FROM hw3_bike_clean
    WHERE NOT holiday
      AND (
            temperature < 0
            OR temperature > 25
          )
    ORDER BY rented_bike_count DESC
    LIMIT 10;
""", engine)

,date,hour,temperature,rented_bike_count
0,2018-06-21,18,27.8,3418
1,2018-06-20,18,27.0,3384
2,2018-06-22,18,29.3,3365
3,2018-06-08,18,26.2,3309
4,2018-09-10,18,25.9,3298
5,2018-09-17,18,25.3,3277
6,2018-09-12,18,27.0,3256
7,2018-05-28,18,26.0,3245
8,2018-06-25,18,30.7,3238
9,2018-06-05,18,27.1,3227


Записи з високим попитом за екстремальних температур переважно припадають приблизно на 18:00 у теплі звичайні дні. Найвище значення в цій вибірці становить 3 418 оренд за температури 27,8°C. Це показує, що висока температура не обов’язково зменшує попит, а вечірній ефект, пов’язаний із поїздками з роботи, залишається досить сильним.

**Business-question #7:** Який середній попит, якщо виключити години з нульовою кількістю оренд?

In [28]:
pd.read_sql("""
    SELECT
        ROUND(
            AVG(NULLIF(rented_bike_count, 0)),
            2
        ) AS avg_non_zero_rentals
    FROM hw3_bike_clean;
""", engine)

,avg_non_zero_rentals
0,729.16


Якщо виключити записи з нульовою кількістю оренд, середній попит за годину становить 729,16 оренди. Це трохи вище за загальне середнє значення, оскільки години з нульовим попитом більше не зменшують середній показник.

**Business-question #8:** Чи є пропущені важливі погодні характеристики?

In [29]:
pd.read_sql("""
    SELECT
        COUNT(*) FILTER (WHERE temperature IS NULL) AS missing_temperature,
        COUNT(*) FILTER (WHERE humidity IS NULL) AS missing_humidity,
        COUNT(*) FILTER (WHERE wind_speed IS NULL) AS missing_wind_speed,
        COUNT(*) FILTER (WHERE rainfall IS NULL) AS missing_rainfall,
        COUNT(*) FILTER (WHERE snowfall IS NULL) AS missing_snowfall
    FROM hw3_bike_clean;
""", engine)

,missing_temperature,missing_humidity,missing_wind_speed,missing_rainfall,missing_snowfall
0,0,0,0,0,0


У вибраних погодних характеристиках не було знайдено пропущених значень: температура, вологість, швидкість вітру, кількість опадів і снігопад. Тому ці поля є повністю заповненими в очищеному наборі даних.

**Business-question #9:** Чи є оренди в ті моменти, коли система прокату велосипедів не працює?

In [30]:
pd.read_sql("""
    SELECT
        date,
        hour,
        functioning_day,
        rented_bike_count
    FROM hw3_bike_clean
    WHERE functioning_day IS DISTINCT FROM TRUE
      AND rented_bike_count > 0
    ORDER BY rented_bike_count DESC;
""", engine)

,date,hour,functioning_day,rented_bike_count


Не було знайдено жодного додатного значення кількості оренд у ті моменти, коли система прокату велосипедів була позначена як непрацююча. Це логічно відповідає значенню поля `functioning_day`.

**Business-question #10:** Які категорії сезонів, що починаються з літери «S», присутні в даних?

In [31]:
pd.read_sql(
    text("""
        SELECT DISTINCT
            seasons
        FROM hw3_bike_clean
        WHERE seasons ILIKE :pattern
        ORDER BY seasons;
    """),
    engine,
    params={"pattern": "S%"}
)

,seasons
0,Spring
1,Summer


Категорії сезонів, які починаються з літери «S», — це `Spring` та `Summer`. Цей запит демонструє пошук за шаблоном без урахування регістру за допомогою ILIKE, а також видалення повторюваних значень категорій за допомогою DISTINCT.

## Завдання 6. Reflection
Набір даних Seoul Bike загалом має хорошу якість. Дублікатів за бізнес-ключем `(date, hour)` не було знайдено, а і в staging-, і в clean-таблицях міститься по 8 760 рядків. У перевірених колонках частка NULL становила 0%, також не було знайдено порожніх рядків, позначок `?` або неочікуваних категоріальних значень. Категоріальні поля містили тільки очікувані значення для сезонів, святкових днів і робочих/неробочих днів.

Для ML-моделі я б використовувала часові ознаки, наприклад годину та сезон, погодні показники — температуру, вологість, швидкість вітру, кількість опадів і снігопадів, а також такі ознаки, як `holiday` і `functioning_day`. Технічне поле `record_id` я б не використовувала, оскільки воно не містить корисної інформації для прогнозування.

Якщо мета — прогнозувати майбутній попит на велосипеди, то `rented_bike_count` має бути цільовою змінною і не повинен використовуватися як вхідна ознака для того самого періоду прогнозування. Фактичні майбутні погодні значення також можуть спричинити витік даних, якщо вони невідомі на момент прогнозу. У production у такому випадку краще використовувати прогноз погоди.

Щоб покращити data pipeline, я б додала автоматичну перевірку схеми, перевірки допустимих значень, пошук дублікатів, контроль кількості рядків, а також сповіщення про неочікувані категоріальні значення або пропущені дані. Парсинг дат і приведення значень до Boolean-типу також варто виконувати однаково під час завантаження даних.

Природним завданням машинного навчання для цього набору даних є прогнозування попиту або регресія, де мета полягає в тому, щоб передбачити кількість орендованих велосипедів у певну майбутню годину.